# 01 · METR-LA data preprocessing

Runs on **Google Colab** or **locally**.

- **Colab:** mounts Google Drive and reads data from `MyDrive/CMPE258_TrafficGNN/raw/` (add a shortcut to the shared team folder under that name).
- **Local:** reads data from `<repo>/data/raw/`. Set `TRAFFIC_DATA_ROOT` to use another folder.

Required raw files: `metr-la.h5` (speeds) and `adj_mx.pkl` (road-distance adjacency).

Outputs written to `processed/` next to `raw/`, shared by every model notebook:

| File | Contents |
|------|----------|
| `metr_la_processed.npz` | raw speeds, normalized + imputed inputs, observed mask, timestamps, sensor ids, distance adjacency |
| `scalers.json` | per-sensor mean / std fitted on the training split |
| `meta.json` | split indices and timestamps, missing-value statistics |

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/mansiguptacs/CMPE_258_TrafficForecastingUsing_GraphNN.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/CMPE_258_TrafficForecastingUsing_GraphNN")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tables", "pyyaml"], check=True)
else:
    repo = Path.cwd()
    while not (repo / "configs" / "metr_la.yaml").exists():
        if repo.parent == repo:
            raise RuntimeError("Start this notebook from inside the project repository")
        repo = repo.parent

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print("Colab" if IN_COLAB else "Local", "| repo:", repo)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, raw_dir, processed_dir
from src.data.load import load_speed, load_adjacency
from src.data.missing import observed_mask, missing_stats
from src.data.pipeline import prepare, load_processed, make_datasets

cfg = load_config()
RAW, PROCESSED = raw_dir(cfg), processed_dir(cfg)
for name in (cfg["paths"]["speed_file"], cfg["paths"]["adjacency_file"]):
    if not (RAW / name).exists():
        raise FileNotFoundError(f"Missing {RAW / name}. Copy it from the team Drive folder.")
print("raw:      ", RAW)
print("processed:", PROCESSED)

## 1. Raw data overview

In [ ]:
speeds, timestamps, sensor_ids = load_speed(RAW / cfg["paths"]["speed_file"])
adj_ids, adj = load_adjacency(RAW / cfg["paths"]["adjacency_file"])

print(f"speeds:     {speeds.shape}  (timesteps x sensors)")
print(f"time range: {timestamps[0]} -> {timestamps[-1]}  ({(timestamps[-1] - timestamps[0]).days + 1} days)")
print(f"interval:   {pd.Series(timestamps).diff().mode()[0]}")
print(f"adjacency:  {adj.shape}, {(adj > 0).sum()} weighted edges, same sensor order: {adj_ids == sensor_ids}")

## 2. Missing values

METR-LA records a sensor dropout as speed `0`. These readings are treated as missing:
they are excluded when fitting the normalization, forward-filled in model **inputs** (past values only),
and masked out of losses and metrics as **targets**.

In [ ]:
observed = observed_mask(speeds, cfg["normalization"]["missing_value"])
stats = missing_stats(observed)
per_sensor = np.array(stats["per_sensor_pct"])

print(f"overall missing:            {stats['overall_pct']:.2f}%")
print(f"timesteps with all missing: {stats['fully_missing_steps']}")
print(f"sensors with >20% missing:  {(per_sensor > 20).sum()}")

daily = pd.Series((~observed).mean(axis=1) * 100, index=timestamps).resample("D").mean()
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].bar(range(len(per_sensor)), np.sort(per_sensor)[::-1], width=1.0)
axes[0].set(title="Missing readings per sensor (sorted)", xlabel="sensor rank", ylabel="% missing")
daily.plot(ax=axes[1])
axes[1].set(title="Missing readings per day (all sensors)", ylabel="% missing", xlabel="")
plt.tight_layout(); plt.show()

In [ ]:
sensor = int(np.argsort(per_sensor)[len(per_sensor) // 2])
week = slice(0, 7 * 24 * 12)
s = pd.Series(speeds[week, sensor], index=timestamps[week])
gaps = ~observed[week, sensor]

fig, ax = plt.subplots(figsize=(14, 3.5))
s.where(~gaps).plot(ax=ax, lw=0.8, label="speed")
ax.scatter(s.index[gaps], np.zeros(gaps.sum()), color="red", s=4, label="missing (0)")
ax.set(title=f"Sensor {sensor_ids[sensor]}: first week", ylabel="mph")
ax.legend(); plt.tight_layout(); plt.show()

## 3. Preprocess and save

Chronological split (no shuffling), per-sensor z-score fitted on the training split only,
forward-fill imputation for inputs. Settings come from `configs/metr_la.yaml`.

In [ ]:
meta = prepare(cfg)

pd.DataFrame({
    split: {
        "start": meta["split_time"][split][0],
        "end": meta["split_time"][split][1],
        "timesteps": b - a,
        "missing %": round(meta["missing_pct_by_split"][split], 2),
    }
    for split, (a, b) in meta["split_index"].items()
}).T

## 4. Verify the processed data

In [ ]:
from torch.utils.data import DataLoader

data = load_processed(cfg)
a, b = data["meta"]["split_index"]["train"]
z = data["inputs"][a:b][data["observed"][a:b]]
print(f"train inputs after z-score: mean {z.mean():.3f}, std {z.std():.3f}")
print(f"NaNs in inputs: {int(np.isnan(data['inputs']).sum())}")

datasets = make_datasets(cfg, data)
for split, ds in datasets.items():
    print(f"{split:5s} windows: {len(ds):6d}")

x, y, y_mask = next(iter(DataLoader(datasets["train"], batch_size=64, shuffle=True)))
print("batch x:", tuple(x.shape), " y:", tuple(y.shape), " y_mask:", tuple(y_mask.shape))

Batches are `x [batch, 12, 207, 1]` (normalized history), `y [batch, 12, 207]` (future speeds in mph)
and `y_mask` (True where the target is a real reading). Metrics are reported at horizons 3, 6 and 12
(15, 30 and 60 minutes). Other notebooks load these files with `load_processed` and `make_datasets`
instead of re-running this notebook.